In [ ]:
# pip install openai python-dotenv
# pip install -e "/path/to/runledger/packages/sdk[openai]"
#
# Create a .env file in this directory (copy .env.example) before running.
# Key variables:
#   RUNLEDGER_API_KEY   — your workspace API key
#   RUNLEDGER_BASE_URL  — http://localhost:8000  (local Docker stack)
#   RUNLEDGER_LOCAL     — "true" to print events instead of sending to the API
#   OLLAMA_BASE_URL     — http://localhost:11434/v1
#   OLLAMA_MODEL        — e.g. llama3.2

In [ ]:
from __future__ import annotations

import os

import openai
from dotenv import load_dotenv

# Load .env from this directory (create one by copying .env.example)
load_dotenv()

In [11]:
from runledger_sdk import RunLedger

In [ ]:
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434/v1")
MODEL           = os.getenv("OLLAMA_MODEL", "llama3.2")

# RUNLEDGER_LOCAL=true  → prints events as JSON to stdout (no running API needed)
# RUNLEDGER_LOCAL=false → sends events to your RunLedger container
LOCAL_MODE = os.getenv("RUNLEDGER_LOCAL", "false").lower() in ("1", "true", "yes")

print(f"Ollama base URL : {OLLAMA_BASE_URL}")
print(f"Ollama model    : {MODEL}")
print(f"RunLedger URL   : {os.getenv('RUNLEDGER_BASE_URL', 'http://localhost:8000')}")
print(f"Local mode      : {LOCAL_MODE}")

In [ ]:
# ── 1. RunLedger client ───────────────────────────────────────────────────────
#
# local=True  → prints events as JSON to stdout (RUNLEDGER_LOCAL=true in .env)
# local=False → sends events to your RunLedger API container
#
# base_url reads from RUNLEDGER_BASE_URL env var (default: http://localhost:8000)
# api_key  reads from RUNLEDGER_API_KEY  env var
rl = RunLedger(
    local=LOCAL_MODE,
    # base_url and api_key are picked up automatically from env vars
)

In [14]:
# ── 2. Instrument — patches openai.OpenAI so every call is captured ───────────
rl.instrument()


In [ ]:
# ── 3. OpenAI client pointed at Ollama ────────────────────────────────────────
#
# Ollama exposes an OpenAI-compatible REST API at /v1.
# api_key can be any non-empty string — Ollama ignores it.
client = openai.OpenAI(
    base_url=OLLAMA_BASE_URL,
    api_key="ollama",
)

In [16]:
# ── Agent: simple multi-turn Q&A ──────────────────────────────────────────────

def run_chat(user_id: str, questions: list[str]) -> None:
    """Send a sequence of questions, maintaining conversation history."""

    history: list[dict[str, str]] = [
        {"role": "system", "content": "You are a concise, helpful assistant."},
    ]

    with rl.context(
        end_user_id=user_id,
        feature_tag="ollama-demo",
        deployment_version="v1.0",
    ) as run_id:
        print(f"\n[RunLedger] run_id={run_id}")
        print(f"[Model]     {MODEL}  ({OLLAMA_BASE_URL})\n")

        for question in questions:
            history.append({"role": "user", "content": question})

            response = client.chat.completions.create(
                model=MODEL,
                messages=history,  # type: ignore[arg-type]
                temperature=0.7,
            )

            answer = response.choices[0].message.content or ""
            history.append({"role": "assistant", "content": answer})

            print(f"Q: {question}")
            print(f"A: {answer}")
            print()

In [17]:
if __name__ == "__main__":
    run_chat(
        user_id="user-local",
        questions=[
            "What is a large language model? One sentence.",
            "Give me one real-world use case for it.",
            "What is the main cost driver when running these at scale?",
        ],
    )

    # Flush all buffered events before exit
    rl.shutdown()


[RunLedger] run_id=9ddd1b06-6c3f-4673-8c54-d73bb085b643
[Model]     llama3.2  (http://localhost:11434/v1)

Q: What is a large language model? One sentence.
A: A large language model (LLM) is a type of artificial intelligence designed to process and understand human language by analyzing vast amounts of text data, generating responses, and learning patterns in natural language processing.

{"event_type": "run_start", "run_id": "9ddd1b06-6c3f-4673-8c54-d73bb085b643", "started_at": "2026-03-01T04:04:39.042621+00:00", "end_user_id": "user-local", "feature_tag": "ollama-demo", "deployment_version": "v1.0"}
{"event_type": "provider_call", "run_id": "9ddd1b06-6c3f-4673-8c54-d73bb085b643", "provider": "openai", "model": "llama3.2", "latency_ms": 1183, "status": "success", "input_tokens": 43, "output_tokens": 41}
{"event_type": "run_end", "run_id": "9ddd1b06-6c3f-4673-8c54-d73bb085b643", "status": "succeeded", "ended_at": "2026-03-01T04:04:40.226740+00:00", "total_input_tokens": 43, "total_out

{"event_type": "provider_call", "run_id": "9ddd1b06-6c3f-4673-8c54-d73bb085b643", "provider": "openai", "model": "llama3.2", "latency_ms": 1226, "status": "success", "input_tokens": 103, "output_tokens": 49}
{"event_type": "run_end", "run_id": "9ddd1b06-6c3f-4673-8c54-d73bb085b643", "status": "succeeded", "ended_at": "2026-03-01T04:04:41.454158+00:00", "total_input_tokens": 103, "total_output_tokens": 49}
{"event_type": "run_start", "run_id": "9ddd1b06-6c3f-4673-8c54-d73bb085b643", "started_at": "2026-03-01T04:04:41.454293+00:00", "end_user_id": "user-local", "feature_tag": "ollama-demo", "deployment_version": "v1.0"}
{"event_type": "provider_call", "run_id": "9ddd1b06-6c3f-4673-8c54-d73bb085b643", "provider": "openai", "model": "llama3.2", "latency_ms": 1021, "status": "success", "input_tokens": 173, "output_tokens": 47}
{"event_type": "run_end", "run_id": "9ddd1b06-6c3f-4673-8c54-d73bb085b643", "status": "succeeded", "ended_at": "2026-03-01T04:04:42.476242+00:00", "total_input_tokens